# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Agha314/FLyRank-Task-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:**
The original table, `fact_content_daily_performance`, has one row for each day, page, and client. In other words, each row represents the performance of one page for one client on one specific day.

For the model, I combine these daily records into one row for each page and client for the whole month. So, the model works with **one page per client for March 2026**.

**Time Window:**
The data used is from **March 2026 (March 1 to March 31)**. This month was chosen because it is in the middle of the available data period.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Features (Honest X), aggregated per page over March:
- total_gsc_impressions_mar = SUM(gsc_impressions) — a count of past search impressions,
  fully known by month-end.
- total_gsc_clicks_mar = SUM(gsc_clicks) — past clicks, same reasoning.
- avg_gsc_position_mar = AVG(gsc_avg_position) — historical average ranking position
  over the month, known at decision time.
- ctr_mar = total_gsc_clicks_mar / total_gsc_impressions_mar — derived from the two
  historical counts above, computable at month-end.
- days_gsc_available = COUNT(*) WHERE gsc_data_available IS TRUE — how many days of
  trustworthy GSC data this page actually had in March, known at decision time.

Label (Proxy Y): is_declining. Split March into first half (days 1–15) and second half
(days 16–31); sum gsc_clicks per page in each half. is_declining = 1 if second-half
clicks < first-half clicks, else 0. This is a within-month proxy, not a true forward-
looking trend — a real trend label needs multiple months, which this single partition
doesn't have (noted in Section 4).

Excluded: second_half_clicks (and any other second-half-only aggregate).

Why: this is
the exact quantity the label is computed from — including it as a feature would let the
model see the answer directly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_Token')}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# con.sql(f"DESCRIBE SELECT * FROM {REL}").show()
con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM {REL}").show()

con.sql(f'SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n FROM {REL} GROUP BY report_date, client_hash_id, content_hash_id HAVING COUNT(*) > 1 LIMIT 5')
con.sql(f'SELECT COUNT(*) AS n_available FROM {REL} WHERE gsc_data_available IS TRUE')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┐
│ n_available │
│    int64    │
├─────────────┤
│     3611061 │
└─────────────┘

In [4]:
# Pull the feature frame into pandas
df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS total_gsc_impressions_mar,
    SUM(gsc_clicks) AS total_gsc_clicks_mar,
    AVG(gsc_avg_position) AS avg_gsc_position_mar,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc_available,
    SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_mar,
    SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') AS first_half_clicks,
    SUM(gsc_clicks) FILTER (WHERE report_date > DATE '2026-03-15') AS second_half_clicks,
    CASE WHEN SUM(gsc_clicks) FILTER (WHERE report_date > DATE '2026-03-15')
              < SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15')
         THEN 1 ELSE 0 END AS is_declining
FROM {REL}
GROUP BY client_hash_id, content_hash_id
""").df()

from sklearn.tree import DecisionTreeClassifier, export_text
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

honest_features = ["total_gsc_impressions_mar", "total_gsc_clicks_mar",
                    "avg_gsc_position_mar", "days_gsc_available", "ctr_mar"]
y = df["is_declining"].values

# Honest model
X_honest = df[honest_features].replace([np.inf, -np.inf], np.nan).fillna(0)
honest_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_honest, y)
print(f"Honest Precision@50: {precision_at_k(honest_tree.predict_proba(X_honest)[:,1], y, 50):.3f}")

# Leaky model — add second_half_clicks on purpose
X_leaky = df[honest_features + ["second_half_clicks"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' Precision@50: {precision_at_k(leaky_tree.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- watch this jump")
print(export_text(leaky_tree, feature_names=honest_features + ["second_half_clicks"]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest Precision@50: 0.380
'Leaky' Precision@50: 1.000  <- watch this jump
|--- ctr_mar <= 0.00
|   |--- class: 0
|--- ctr_mar >  0.00
|   |--- second_half_clicks <= 0.50
|   |   |--- class: 1
|   |--- second_half_clicks >  0.50
|   |   |--- class: 1



Adding second_half_clicks pushes Precision@50 from 0.380 to 1.000 — the model is reading the label's own ingredients, not learning a real pattern. second_half_clicks is excluded from the final feature set for this reason

total_gsc_impressions_mar — knowable at decision moment because it's a sum of impressions already observed by the end of March; nothing here depends on future data.

total_gsc_clicks_mar — knowable at decision moment because it's a sum of clicks already observed within March, fully computable using only March's rows.

avg_gsc_position_mar — knowable at decision moment because it's an average of ranking positions recorded during March; no information from beyond the month is used.

days_gsc_available — knowable at decision moment because it's a count of days within March where GSC data was actually available; entirely derived from March's own rows.

ctr_mar — knowable at decision moment because it's derived from total_gsc_clicks_mar and total_gsc_impressions_mar, both of which are themselves fully computed from March data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This March slice treats every client's page equally, but clients' actual tracking history differs — some have years of GSC/GA4 data, others started very recently (see dim_clients.gsc_data_start/ga4_data_start). So 'one month of data' means very different things for different clients, and trend-based labels may be less reliable for newly-tracked clients.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.